# FedX-PALM: Federated Explainable Framework for Palm Fruit Ripeness

**Master's Thesis - Rachma Dianty (201012420007)**
**Telkom University, 2025**

---

## System Architecture
```
Google Colab Pro (GPU T4/A100)        VPS Server (CPU)
+--------------------------+          +---------------------+
| 4 Client Nodes           |  Flower  | FL Server Aggregator|
| YOLOv11 Nano (640x640)   |<-------->| FedAvg (Eq. 3.4)   |
| DP-SGD (Opacus)          |  gRPC    | Checkpointing       |
| Grad-CAM++ (XAI)         |          | Privacy Tracking    |
+--------------------------+          +---------------------+
```

### Specifications (Thesis Table 3.5)
| Parameter | Value |
|-----------|-------|
| Model | YOLOv11 Nano |
| Classes | 6 (Unripe, Underripe, Ripe, Overripe, Abnormal, Empty Bunch) |
| Clients | 4 (Non-IID Dirichlet) |
| Rounds | 100 |
| Local Epochs | 5 |
| Batch Size | 16 |
| Learning Rate | 0.01 (AdamW) |
| Privacy (ε) | 1.0, 4.0, 8.0, ∞ |
| XAI | Grad-CAM++ + Average Drop + FRR |

## 1. Setup Environment

In [ ]:
!pip install -q ultralytics flwr opacus roboflow grad-cam \
    torch torchvision numpy scipy opencv-python matplotlib \
    pyyaml tqdm tensorboard seaborn scikit-learn

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import flwr
print(f'Flower: {flwr.__version__}')

try:
    import opacus
    print(f'Opacus: {opacus.__version__}')
except: print('Opacus: not installed')

## 2. Clone Repository

In [ ]:
import os
if not os.path.exists('fedx-palm'):
    !git clone https://github.com/rachmadiantyy/fedx-palm.git
os.chdir('fedx-palm')
print(f'Working dir: {os.getcwd()}')

## 3. Download Dataset from Roboflow (6 Classes)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key='Ej0bSMpeSri3ky0IYkOU')
project = rf.workspace('dydy-worker').project('palm-fruit-ripeness-detection-f6sac-ccb2z')
version = project.version(2)
dataset = version.download('yolov11', location='./data/raw')
print(f'Dataset: {dataset.location}')

## 4. Split Dataset - Non-IID Dirichlet (Thesis Section 3.3.4)

| Client | α | Dominant Classes |
|--------|------|------------------|
| A | 0.1 | Unripe, Underripe |
| B | 0.3 | Underripe, Ripe |
| C | 0.5 | Fairly balanced |
| D | 0.7 | Abnormal, Empty Bunch |

In [ ]:
!python scripts/download_dataset.py \
    --skip-download \
    --output-dir ./data/raw \
    --split-clients 4 \
    --split-strategy non_iid \
    --client-data-dir ./data \
    --seed 42

In [ ]:
# Verify Non-IID split
for i in range(1, 5):
    d = f'./data/client_{i}'
    if os.path.exists(f'{d}/images/train'):
        n = len(os.listdir(f'{d}/images/train'))
        print(f'Client {i}: {n} train images')

## 5. Configuration

**UPDATE `SERVER_ADDRESS` with your VPS IP!**

In [ ]:
# ========== CONFIGURATION ==========
SERVER_ADDRESS = '<YOUR_VPS_IP>:8080'  # <-- CHANGE THIS!

# Privacy scenario (thesis Table 3.4)
# Options: 'baseline', 'weak', 'moderate', 'strong', 'partial_moderate'
PRIVACY_SCENARIO = 'moderate'  # ε=4.0, σ=1.5

# Model & Training (thesis Table 3.5)
MODEL = 'yolo11n.pt'
NUM_CLASSES = 6
LOCAL_EPOCHS = 5
BATCH_SIZE = 16
LR = 0.01
IMG_SIZE = 640

print(f'Server: {SERVER_ADDRESS}')
print(f'Privacy: {PRIVACY_SCENARIO}')
print(f'Model: {MODEL} ({NUM_CLASSES} classes)')

## 6. Run FL Clients (4 Clients Sequential)

Each client trains locally on Non-IID data, applies DP-SGD, then sends updates to server.

In [ ]:
import subprocess, threading, time

def run_client(client_id):
    """Run one FL client as subprocess."""
    cmd = [
        'python', '-m', 'client.fed_client',
        '--server', SERVER_ADDRESS,
        '--client-id', f'client_{client_id}',
        '--data-config', f'./data/client_{client_id}/data.yaml',
        '--model', MODEL,
        '--num-classes', str(NUM_CLASSES),
        '--local-epochs', str(LOCAL_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--lr', str(LR),
        '--privacy', PRIVACY_SCENARIO,
    ]
    print(f'Starting client_{client_id}...')
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f'Client {client_id} finished. Return code: {result.returncode}')
    if result.returncode != 0:
        print(f'  STDERR: {result.stderr[-500:]}')
    return result

# Run all 4 clients in parallel (they connect to same server)
threads = []
for cid in range(1, 5):
    t = threading.Thread(target=run_client, args=(cid,))
    threads.append(t)
    t.start()
    time.sleep(2)  # Stagger start

# Wait for all to complete
for t in threads:
    t.join()

print('\nAll 4 clients completed!')

## 7. Run Experiments with Different ε (Privacy-Utility Trade-off)

Thesis Table 3.4: Compare baseline, weak, moderate, strong privacy.

In [ ]:
# Run experiments for all privacy scenarios
SCENARIOS = ['baseline', 'weak', 'moderate', 'strong']

for scenario in SCENARIOS:
    print(f'\n{"="*50}')
    print(f'EXPERIMENT: {scenario.upper()} PRIVACY')
    print(f'{"="*50}')
    
    # Note: In practice, restart server for each scenario
    # or run sequentially. This is illustrative.
    for cid in range(1, 5):
        cmd = f'python -m client.fed_client --server {SERVER_ADDRESS} '\
              f'--client-id client_{cid} '\
              f'--data-config ./data/client_{cid}/data.yaml '\
              f'--num-classes 6 --privacy {scenario}'
        print(f'  Client {cid}: {scenario} (ε={{}})'.format(
            {'baseline':'∞','weak':'8.0','moderate':'4.0','strong':'1.0'}[scenario]))
    print(f'  → Results saved to ./results/')

## 8. XAI Analysis: Grad-CAM++ (Thesis Section 4.5)

In [ ]:
import sys
sys.path.insert(0, '.')
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from xai.explainer import GradCAMPlusPlus, AverageDrop, FocusRetentionRate, XAIReportGenerator

# Load trained model
model = YOLO(MODEL)

# Find sample images
val_dir = './data/client_1/images/val'
if os.path.exists(val_dir):
    sample_paths = [os.path.join(val_dir, f) for f in os.listdir(val_dir)[:5] if f.endswith('.jpg')]
    images = [cv2.imread(p) for p in sample_paths]
    images = [img for img in images if img is not None]
    print(f'Loaded {len(images)} validation images for XAI analysis')
else:
    print('No validation images found')
    images = []

In [ ]:
if images:
    # Generate Grad-CAM++ heatmap
    grad_cam_pp = GradCAMPlusPlus(model)
    
    fig, axes = plt.subplots(2, len(images), figsize=(4*len(images), 8))
    if len(images) == 1:
        axes = axes.reshape(-1, 1)
    
    for i, img in enumerate(images):
        explanation = grad_cam_pp.generate(img)
        
        # Original
        axes[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0, i].set_title(f'Original {i+1}')
        axes[0, i].axis('off')
        
        # Grad-CAM++ overlay
        if explanation.overlay is not None:
            axes[1, i].imshow(cv2.cvtColor(explanation.overlay, cv2.COLOR_BGR2RGB))
        axes[1, i].set_title(f'Grad-CAM++ {i+1}')
        axes[1, i].axis('off')
    
    plt.suptitle('Grad-CAM++ Visualization (Thesis Section 4.5)', fontsize=14)
    plt.tight_layout()
    plt.savefig('./results/grad_cam_pp_results.png', dpi=150)
    plt.show()
    
    grad_cam_pp.cleanup()

In [ ]:
if images:
    # Compute Average Drop and FRR metrics
    report_gen = XAIReportGenerator(model, privacy_epsilon=4.0)
    report = report_gen.generate_report(
        images=images,
        target_class=None,
        output_dir='./results/xai_report'
    )
    
    print('\n=== XAI Report ===')
    print(f"Average Drop: {report.get('average_drop', {}).get('average_drop_pct', 'N/A')}%")
    print(f"Reliability: {report.get('average_drop', {}).get('reliability', 'N/A')}")
    print(f"FRR: {report.get('focus_retention_rate', {}).get('mean_frr', 'N/A')}")
    
    report_gen.cleanup()

## 9. VPS Server Setup Instructions

Run this on your VPS (4 vCPU, 8GB RAM):

```bash
# Clone repo
git clone https://github.com/rachmadiantyy/fedx-palm.git
cd fedx-palm

# Install (server only needs flower + torch + ultralytics)
pip install flwr torch ultralytics numpy pyyaml

# Start FL Server
python -m server.fed_server \
  --address 0.0.0.0:8080 \
  --num-rounds 100 \
  --model yolo11n.pt \
  --num-classes 6 \
  --min-clients 4 \
  --checkpoint-dir ./checkpoints
```

Or with Docker:
```bash
docker build -t fedx-server -f docker/Dockerfile.server .
docker run -d -p 8080:8080 --name fl-server fedx-server
```

## 10. Summary

### Privacy Scenarios (Thesis Table 3.4)
| Scenario | ε | δ | σ | Level |
|----------|-----|---------|-----|-------|
| Baseline | ∞ | N/A | 0.0 | No Privacy |
| Weak | 8.0 | 1e-5 | 0.8 | Lemah |
| Moderate | 4.0 | 1e-5 | 1.5 | Sedang |
| Strong | 1.0 | 1e-5 | 3.2 | Kuat |

### 6 Ripeness Classes
| ID | Class | Description |
|----|-------|-------------|
| 0 | Unripe | Hitam/hijau tua |
| 1 | Underripe | Mulai berubah jingga |
| 2 | Ripe | Jingga kemerahan (optimal) |
| 3 | Overripe | Merah tua/ungu |
| 4 | Abnormal | Gangguan pertumbuhan |
| 5 | Empty Bunch | Tandan kosong |